In [15]:
import tkinter as tk
from tkinter import ttk
import tkinter.messagebox

class Book:
    __books_read = 0

    def __init__(self, title, author, total_pages, category):
        self.__title = title
        self.__author = author
        self.__total_pages = total_pages
        self.__category = category
        self.__rating = None
        self.__current_page = 0
        self.__finished = False
        self.__ratings_history = []

    def get_title(self):
        return self.__title

    def get_author(self):
        return self.__author

    def get_total_pages(self):
        return self.__total_pages

    def get_category(self):
        return self.__category

    def get_rating(self):
        return self.__rating

    def get_current_page(self):
        return self.__current_page

    def is_finished(self):
        return self.__finished

    def get_ratings_history(self):
        return self.__ratings_history.copy()

    def next_page(self):
        if self.__current_page < self.__total_pages:
            self.__current_page += 1
            if self.__current_page >= self.__total_pages and not self.__finished:
                self.__finished = True
                Book.__books_read += 1
                return True
        return False

    def previous_page(self):
        if self.__current_page > 0:
            self.__current_page -= 1

    def rate_book(self, rating):
        if 1 <= rating <= 5:
            self.__rating = rating
            self.__ratings_history.append(rating)
            return True
        return False

    def average_rating(self):
        if not self.__ratings_history:
            return 0
        return sum(self.__ratings_history) / len(self.__ratings_history)

    def progress_percent(self):
        return (self.__current_page / self.__total_pages) * 100

    @classmethod
    def books_read_count(cls):
        return cls.__books_read


class Person:
    def __init__(self, password, name, ID):
        self.__password = password
        self.__name = name
        self.__ID = ID

    def get_name(self):
        return self.__name

    def get_ID(self):
        return self.__ID

    def login(self, entered_password):
        return self.__password == entered_password


class User(Person):
    def __init__(self, password, name, ID):
        super().__init__(password, name, ID)
        self.__books = []
        self.__current_book_title = None

    def get_books(self):
        return self.__books.copy()

    def get_current_book_title(self):
        return self.__current_book_title

    def start_reading_book(self, book):
        if book.get_title() not in self.__books:
            self.__books.append(book.get_title())
        self.__current_book_title = book.get_title()

    def add_book(self, book):
        if book.get_title() not in self.__books:
            self.__books.append(book.get_title())
        if not self.__current_book_title:
            self.__current_book_title = book.get_title()

    def select_book(self, title):
        if title in self.__books:
            self.__current_book_title = title
            return True
        return False

    def remove_book_from_list(self, book_title):
        if book_title in self.__books:
            self.__books.remove(book_title)

    def clear_current_book(self):
        self.__current_book_title = None

    def books_read_count(self, system_books):
        count = 0
        for book_title in self.__books:
            for book in system_books:
                if book.get_title() == book_title and book.is_finished():
                    count += 1
        return count


class Admin(Person):
    pass


class LibrarySystem:
    def __init__(self):
        self.__books = []
        self.__users = []
        self.__admins = []
        self.__logged_in_user = None

    def get_books(self):
        return self.__books.copy()

    def get_users(self):
        return self.__users.copy()

    def get_admins(self):
        return self.__admins.copy()

    def get_logged_in_user(self):
        return self.__logged_in_user

    def add_user(self, user):
        self.__users.append(user)

    def add_admin(self, admin):
        self.__admins.append(admin)

    def add_book(self, book):
        self.__books.append(book)

    def remove_book(self, book):
        if book in self.__books:
            self.__books.remove(book)
            for user in self.__users:
                if book.get_title() in user.get_books():
                    user.remove_book_from_list(book.get_title())
                if user.get_current_book_title() == book.get_title():
                    user.clear_current_book()

    def find_book_by_title(self, title):
        for book in self.__books:
            if book.get_title() == title:
                return book
        return None

    def login(self, ID, password):
        ID = str(ID).strip()
        for u in self.__users + self.__admins:
            if str(u.get_ID()).strip() == ID and u.login(password):
                self.__logged_in_user = u
                return True
        return False

    def logout(self):
        self.__logged_in_user = None


system = LibrarySystem()
system.add_admin(Admin("admin123", "Reem", 1))

BG_COLOR = "#051115"
BUTTON_COLOR = "#40E0D0"
TEXT_COLOR = "white"
LABEL_COLOR = "#40E0D0"

root = tk.Tk()
root.title("Library System")
root.geometry("1000x700")
root.configure(bg=BG_COLOR)

def create_label(parent, text, font_size=24, pady=10, bold=True, bg=BG_COLOR, fg=LABEL_COLOR):
    font_style = ("Helvetica", font_size, "bold" if bold else "normal")
    lbl = tk.Label(parent, text=text, font=font_style, bg=bg, fg=fg)
    lbl.pack(pady=pady)
    return lbl

def create_button(parent, text, command, width=20, font_size=20, pady=10, bg_color=BUTTON_COLOR, fg_color=TEXT_COLOR, side=None):
    btn = tk.Button(parent, text=text, font=("Helvetica", font_size, "bold"),
                    bg=bg_color, fg=fg_color, width=width, command=command)
    if side:
        btn.pack(side=side, pady=pady)
    else:
        btn.pack(pady=pady)
    return btn

def create_entry(parent, font_size=18, show=None, pady=5):
    entry = tk.Entry(parent, font=("Helvetica", font_size), show=show)
    entry.pack(pady=pady)
    return entry

def login_screen():
    for w in root.winfo_children():
        w.destroy()
    create_label(root, "Welcome to Our Library!", font_size=42, pady=50)
    create_button(root, "Login", login_form_screen, width=15)
    create_button(root, "SignUp", signup_screen, width=15)

def login_form_screen():
    for w in root.winfo_children():
        w.destroy()
    create_label(root, "Login", font_size=36, pady=30)
    create_label(root, "ID:", font_size=24)
    id_entry = create_entry(root)
    create_label(root, "Password:", font_size=24)
    pw_entry = create_entry(root, show="*")
    info_label = create_label(root, "", font_size=20)

    def attempt_login():
        try:
            ID = int(id_entry.get())
        except:
            info_label.config(text="Invalid ID!")
            return
        if system.login(ID, pw_entry.get()):
            welcome_screen()
        else:
            info_label.config(text="Wrong ID or Password!")

    create_button(root, "Login", attempt_login)
    create_button(root, "Back", login_screen)

def signup_screen():
    for w in root.winfo_children():
        w.destroy()
    create_label(root, "Sign Up", font_size=36, pady=30)
    create_label(root, "Name:", font_size=24)
    name_entry = create_entry(root)
    create_label(root, "ID:", font_size=24)
    id_entry = create_entry(root)
    create_label(root, "Password:", font_size=24)
    pw_entry = create_entry(root, show="*")
    info_label = create_label(root, "", font_size=20)

    def create_user_action():
        name = name_entry.get()
        pw = pw_entry.get()
        try:
            ID = int(id_entry.get())
        except:
            info_label.config(text="Invalid data!")
            return
        for u in system.get_users() + system.get_admins():
            if u.get_ID() == ID:
                info_label.config(text="ID already exists!")
                return
        if not name or not pw:
            info_label.config(text="Invalid data!")
            return
        user = User(pw, name, ID)
        system.add_user(user)
        system.login(ID, pw)
        welcome_screen()

    create_button(root, "Create account", create_user_action)
    create_button(root, "Back", login_screen)

def welcome_screen():
    for w in root.winfo_children():
        w.destroy()
    user = system.get_logged_in_user()
    create_label(root, f"Welcome {user.get_name()}!", font_size=36, pady=30)

    if isinstance(user, Admin):
        create_button(root, "Add Book", add_book_screen)
        create_button(root, "Remove Book", remove_book_screen)

    if isinstance(user, User):
        read_count = user.books_read_count(system.get_books())
        create_label(root, f"You have read {read_count} books.", font_size=22)

        current_title = user.get_current_book_title()
        if current_title:
            current_book = system.find_book_by_title(current_title)
            if current_book:
                create_label(root, f"Currently reading: {current_book.get_title()}", font_size=18, pady=5)
                create_button(root, "Continue Reading", lambda: start_reading(current_book))

        create_button(root, "Select Book to Read", categories_screen)

    create_button(root, "Logout", lambda: [system.logout(), login_screen()], pady=50)

def add_book_screen():
    for w in root.winfo_children():
        w.destroy()
    create_label(root, "Add Book", font_size=36, pady=30)
    create_label(root, "Title:", font_size=20)
    title_entry = create_entry(root)
    create_label(root, "Author:", font_size=20)
    author_entry = create_entry(root)
    create_label(root, "Pages:", font_size=20)
    pages_entry = create_entry(root)
    create_label(root, "Category:", font_size=20)
    category_entry = create_entry(root)
    info_label = create_label(root, "", font_size=20)

    def add_book_action():
        title = title_entry.get().strip()
        author = author_entry.get().strip()
        category = category_entry.get().strip()
        try:
            pages = int(pages_entry.get())
        except:
            pages = None
        if not title or not author or not category or not pages or pages <= 0:
            info_label.config(text="Invalid data!")
            return
        book = Book(title, author, pages, category)
        system.add_book(book)
        info_label.config(text=f"Book '{book.get_title()}' added successfully!")
        for e in [title_entry, author_entry, pages_entry, category_entry]:
            e.delete(0, 'end')

    create_button(root, "Add Book", add_book_action)
    create_button(root, "Back", welcome_screen)

def remove_book_screen():
    for w in root.winfo_children():
        w.destroy()
    create_label(root, "Remove Book", font_size=36, pady=30)
    create_label(root, "Enter book title to remove:", font_size=20)
    title_entry = create_entry(root)
    info_label = create_label(root, "", font_size=20)

    def remove_book_action():
        title = title_entry.get().strip()
        found_books = [b for b in system.get_books() if b.get_title() == title]

        if not found_books:
            info_label.config(text="Book not found!")
            return

        for b in found_books[:]:
            system.remove_book(b)
        info_label.config(text=f"Removed {len(found_books)} book(s)!")
        title_entry.delete(0, 'end')

    create_button(root, "Remove Book", remove_book_action)
    create_button(root, "Back", welcome_screen)

def categories_screen():
    for w in root.winfo_children():
        w.destroy()
    user = system.get_logged_in_user()
    create_label(root, "Categories", font_size=36, pady=20)
    create_button(root, "Search Book", search_screen, width=15)

    books_to_show = system.get_books()
    categories = {}
    for book in books_to_show:
        categories.setdefault(book.get_category(), []).append(book)

    side_frame = tk.Frame(root, bg=BG_COLOR)
    side_frame.pack(side="left", fill="y", padx=20)

    content_frame = tk.Frame(root, bg=BG_COLOR)
    content_frame.pack(side="left", fill="both", expand=True, padx=20)

    canvas = tk.Canvas(content_frame, bg=BG_COLOR, highlightthickness=0)
    scrollbar = tk.Scrollbar(content_frame, command=canvas.yview)
    scrollbar.pack(side="right", fill="y")
    canvas.pack(side="left", fill="both", expand=True)
    canvas.config(yscrollcommand=scrollbar.set)

    books_frame = tk.Frame(canvas, bg=BG_COLOR)
    canvas.create_window((0, 0), window=books_frame, anchor="nw")

    def show_books(cat):
        for w in books_frame.winfo_children():
            w.destroy()

        for b in categories[cat]:
            display_text = f"{b.get_title()} ({b.get_current_page()}/{b.get_total_pages()})"
            create_button(books_frame, display_text,
                         lambda book=b: book_info_screen(book), width=30, bg_color=LABEL_COLOR)

        books_frame.update_idletasks()
        canvas.config(scrollregion=canvas.bbox("all"))

    for cat in categories:
        count = len(categories[cat])
        create_button(side_frame, f"{cat} ({count})", lambda c=cat: show_books(c), width=15, bg_color=LABEL_COLOR)

    create_button(root, "Back", welcome_screen).pack(side="bottom")

def search_screen():
    for w in root.winfo_children():
        w.destroy()
    create_label(root, "Search Book", font_size=30, pady=20)
    entry = create_entry(root)
    result_frame = tk.Frame(root, bg=BG_COLOR)
    result_frame.pack(fill="both", expand=True)

    def do_search():
        for w in result_frame.winfo_children():
            w.destroy()
        query = entry.get().lower()
        found = False

        for b in system.get_books():
            if (query in b.get_title().lower() or
                query in b.get_author().lower() or
                query in b.get_category().lower()):
                create_button(result_frame, f"{b.get_title()} by {b.get_author()}",
                            lambda book=b: book_info_screen(book), width=30,
                            bg_color=LABEL_COLOR)
                found = True

        if not found:
            create_label(result_frame, "No book found.", font_size=20)

    create_button(root, "Search", do_search)
    create_button(root, "Back", categories_screen)

def book_info_screen(book):
    for w in root.winfo_children():
        w.destroy()
    create_label(root, f"{book.get_title()}", font_size=30, pady=10)
    create_label(root, f"Author: {book.get_author()}", font_size=22, pady=5)

    if not book.get_ratings_history():
        create_label(root, "Average Rating: No ratings yet", font_size=22, pady=5)
    else:
        avg = round(book.average_rating())
        stars = "★" * avg + "☆" * (5 - avg)
        create_label(root, f"Average Rating: {stars}", font_size=22, pady=5)

    create_label(root, f"Pages: {book.get_total_pages()}", font_size=20, pady=5)
    create_label(root, f"Category: {book.get_category()}", font_size=20, pady=5)

    def start_reading_handler():
        if isinstance(system.get_logged_in_user(), User):
            system.get_logged_in_user().start_reading_book(book)
        start_reading(book)

    create_button(root, "Start Reading", start_reading_handler)
    create_button(root, "Back", categories_screen)

def start_reading(book):
    for w in root.winfo_children():
        w.destroy()
    page_label = create_label(root, f"Page {book.get_current_page()}/{book.get_total_pages()}", font_size=30, pady=20)

    def next_page_action():
        finished = book.next_page()
        page_label.config(text=f"Page {book.get_current_page()}/{book.get_total_pages()}")
        if finished:
            rating_screen(book)

    def prev_page_action():
        book.previous_page()
        page_label.config(text=f"Page {book.get_current_page()}/{book.get_total_pages()}")

    create_button(root, "Next Page", next_page_action, width=20)
    create_button(root, "Previous Page", prev_page_action, width=20)
    create_button(root, "Back", lambda: book_info_screen(book), width=20)

def rating_screen(book):
    for w in root.winfo_children():
        w.destroy()
    create_label(root, f"Rate '{book.get_title()}'", font_size=30, pady=30)
    stars_frame = tk.Frame(root, bg=BG_COLOR)
    stars_frame.pack(pady=20)
    rating_var = tk.IntVar()
    for i in range(1, 6):
        tk.Radiobutton(stars_frame, text=str(i), variable=rating_var, value=i,
                       font=("Helvetica", 24), bg=BG_COLOR, fg="white",
                       selectcolor=BG_COLOR).pack(side="left", padx=10)

    def submit_rating():
        r = rating_var.get()
        if r:
            book.rate_book(r)
            thanks_screen(book)

    create_button(root, "Submit", submit_rating, width=15)

def thanks_screen(book):
    for w in root.winfo_children():
        w.destroy()
    create_label(root, f"Thanks for rating '{book.get_title()}'!", font_size=30, pady=50)
    create_button(root, "Try Another Book", categories_screen, width=20)

login_screen()
root.mainloop()

In [16]:
import tkinter as tk
from tkinter import ttk

class ReadingProgress:
    def __init__(self, book):
        self.__book = book
        self.__current_page = 0
        self.__finished = False

    def get_book(self):
        return self.__book

    def get_current_page(self):
        return self.__current_page

    def is_finished(self):
        return self.__finished

    def next_page(self):
        if self.__current_page < self.__book.get_total_pages():
            self.__current_page += 1
            if self.__current_page >= self.__book.get_total_pages() and not self.__finished:
                self.__finished = True
                return True
        return False

    def previous_page(self):
        if self.__current_page > 0:
            self.__current_page -= 1

    def progress_percent(self):
        return (self.__current_page / self.__book.get_total_pages()) * 100


class Book:
    __books_read = 0

    def __init__(self, title, author, total_pages, category):
        self.__title = title
        self.__author = author
        self.__total_pages = total_pages
        self.__category = category
        self.__rating = None
        self.__ratings_history = []

    def get_title(self):
        return self.__title

    def get_author(self):
        return self.__author

    def get_total_pages(self):
        return self.__total_pages

    def get_category(self):
        return self.__category

    def get_rating(self):
        return self.__rating

    def get_ratings_history(self):
        return self.__ratings_history.copy()

    def rate_book(self, rating):
        if 1 <= rating <= 5:
            self.__rating = rating
            self.__ratings_history.append(rating)
            return True
        return False

    def average_rating(self):
        if not self.__ratings_history:
            return 0
        return sum(self.__ratings_history) / len(self.__ratings_history)

    @classmethod
    def books_read_count(cls):
        return cls.__books_read


class Person:
    def __init__(self, password, name, ID):
        self.__password = password
        self.__name = name
        self.__ID = ID

    def get_name(self):
        return self.__name

    def get_ID(self):
        return self.__ID

    def login(self, entered_password):
        return self.__password == entered_password


class User(Person):
    def __init__(self, password, name, ID):
        super().__init__(password, name, ID)
        self.__reading_progress = []  # List of ReadingProgress objects
        self.__current_book_title = None

    def get_reading_progress(self):
        return self.__reading_progress.copy()

    def get_current_book_title(self):
        return self.__current_book_title

    def get_progress_for_book(self, book_title):
        for progress in self.__reading_progress:
            if progress.get_book().get_title() == book_title:
                return progress
        return None

    def start_reading_book(self, book):
        # Check if already has progress for this book
        progress = self.get_progress_for_book(book.get_title())
        if not progress:
            progress = ReadingProgress(book)
            self.__reading_progress.append(progress)
        self.__current_book_title = book.get_title()
        return progress

    def add_book(self, book):
        if not self.get_progress_for_book(book.get_title()):
            progress = ReadingProgress(book)
            self.__reading_progress.append(progress)
        if not self.__current_book_title:
            self.__current_book_title = book.get_title()

    def select_book(self, title):
        if self.get_progress_for_book(title):
            self.__current_book_title = title
            return True
        return False

    def remove_book_from_list(self, book_title):
        progress = self.get_progress_for_book(book_title)
        if progress:
            self.__reading_progress.remove(progress)

    def clear_current_book(self):
        self.__current_book_title = None

    def books_read_count(self):
        count = 0
        for progress in self.__reading_progress:
            if progress.is_finished():
                count += 1
        return count


class Admin(Person):
    pass


class LibrarySystem:
    def __init__(self):
        self.__books = []
        self.__users = []
        self.__admins = []
        self.__logged_in_user = None

    def get_books(self):
        return self.__books.copy()

    def get_users(self):
        return self.__users.copy()

    def get_admins(self):
        return self.__admins.copy()

    def get_logged_in_user(self):
        return self.__logged_in_user

    def add_user(self, user):
        self.__users.append(user)

    def add_admin(self, admin):
        self.__admins.append(admin)

    def add_book(self, book):
        self.__books.append(book)

    def remove_book(self, book):
        if book in self.__books:
            self.__books.remove(book)
            for user in self.__users:
                if user.get_progress_for_book(book.get_title()):
                    user.remove_book_from_list(book.get_title())
                if user.get_current_book_title() == book.get_title():
                    user.clear_current_book()

    def find_book_by_title(self, title):
        for book in self.__books:
            if book.get_title() == title:
                return book
        return None

    def login(self, ID, password):
        ID = str(ID).strip()
        for u in self.__users + self.__admins:
            if str(u.get_ID()).strip() == ID and u.login(password):
                self.__logged_in_user = u
                return True
        return False

    def logout(self):
        self.__logged_in_user = None


system = LibrarySystem()
system.add_admin(Admin("admin123", "Reem", 1))

# بقية الكود زي ماهو...

BG_COLOR = "#051115"
BUTTON_COLOR = "#40E0D0"
TEXT_COLOR = "white"
LABEL_COLOR = "#40E0D0"

root = tk.Tk()
root.title("Library System")
root.geometry("1000x700")
root.configure(bg=BG_COLOR)

def create_label(parent, text, font_size=24, pady=10, bold=True, bg=BG_COLOR, fg=LABEL_COLOR):
    font_style = ("Helvetica", font_size, "bold" if bold else "normal")
    lbl = tk.Label(parent, text=text, font=font_style, bg=bg, fg=fg)
    lbl.pack(pady=pady)
    return lbl

def create_button(parent, text, command, width=20, font_size=20, pady=10, bg_color=BUTTON_COLOR, fg_color=TEXT_COLOR, side=None):
    btn = tk.Button(parent, text=text, font=("Helvetica", font_size, "bold"),
                    bg=bg_color, fg=fg_color, width=width, command=command)
    if side:
        btn.pack(side=side, pady=pady)
    else:
        btn.pack(pady=pady)
    return btn

def create_entry(parent, font_size=18, show=None, pady=5):
    entry = tk.Entry(parent, font=("Helvetica", font_size), show=show)
    entry.pack(pady=pady)
    return entry

def login_screen():
    for w in root.winfo_children():
        w.destroy()
    create_label(root, "Welcome to Our Library!", font_size=42, pady=50)
    create_button(root, "Login", login_form_screen, width=15)
    create_button(root, "SignUp", signup_screen, width=15)

def login_form_screen():
    for w in root.winfo_children():
        w.destroy()
    create_label(root, "Login", font_size=36, pady=30)
    create_label(root, "ID:", font_size=24)
    id_entry = create_entry(root)
    create_label(root, "Password:", font_size=24)
    pw_entry = create_entry(root, show="*")
    info_label = create_label(root, "", font_size=20)

    def attempt_login():
        try:
            ID = int(id_entry.get())
        except:
            info_label.config(text="Invalid ID!")
            return
        if system.login(ID, pw_entry.get()):
            welcome_screen()
        else:
            info_label.config(text="Wrong ID or Password!")

    create_button(root, "Login", attempt_login)
    create_button(root, "Back", login_screen)

def signup_screen():
    for w in root.winfo_children():
        w.destroy()
    create_label(root, "Sign Up", font_size=36, pady=30)
    create_label(root, "Name:", font_size=24)
    name_entry = create_entry(root)
    create_label(root, "ID:", font_size=24)
    id_entry = create_entry(root)
    create_label(root, "Password:", font_size=24)
    pw_entry = create_entry(root, show="*")
    info_label = create_label(root, "", font_size=20)

    def create_user_action():
        name = name_entry.get()
        pw = pw_entry.get()
        try:
            ID = int(id_entry.get())
        except:
            info_label.config(text="Invalid data!")
            return
        for u in system.get_users() + system.get_admins():
            if u.get_ID() == ID:
                info_label.config(text="ID already exists!")
                return
        if not name or not pw:
            info_label.config(text="Invalid data!")
            return
        user = User(pw, name, ID)
        system.add_user(user)
        system.login(ID, pw)
        welcome_screen()

    create_button(root, "Create account", create_user_action)
    create_button(root, "Back", login_screen)

def welcome_screen():
    for w in root.winfo_children():
        w.destroy()
    user = system.get_logged_in_user()
    create_label(root, f"Welcome {user.get_name()}!", font_size=36, pady=30)

    if isinstance(user, Admin):
        create_button(root, "Add Book", add_book_screen)
        create_button(root, "Remove Book", remove_book_screen)

    if isinstance(user, User):
        read_count = user.books_read_count()
        create_label(root, f"You have read {read_count} books.", font_size=22)

        current_title = user.get_current_book_title()
        if current_title:
            current_book = system.find_book_by_title(current_title)
            if current_book:
                create_label(root, f"Currently reading: {current_book.get_title()}", font_size=18, pady=5)
                create_button(root, "Continue Reading", lambda: continue_reading(user, current_book))

        create_button(root, "Select Book to Read", categories_screen)

    create_button(root, "Logout", lambda: [system.logout(), login_screen()], pady=50)

def add_book_screen():
    for w in root.winfo_children():
        w.destroy()
    create_label(root, "Add Book", font_size=36, pady=30)
    create_label(root, "Title:", font_size=20)
    title_entry = create_entry(root)
    create_label(root, "Author:", font_size=20)
    author_entry = create_entry(root)
    create_label(root, "Pages:", font_size=20)
    pages_entry = create_entry(root)
    create_label(root, "Category:", font_size=20)
    category_entry = create_entry(root)
    info_label = create_label(root, "", font_size=20)

    def add_book_action():
        title = title_entry.get().strip()
        author = author_entry.get().strip()
        category = category_entry.get().strip()
        try:
            pages = int(pages_entry.get())
        except:
            pages = None
        if not title or not author or not category or not pages or pages <= 0:
            info_label.config(text="Invalid data!")
            return
        book = Book(title, author, pages, category)
        system.add_book(book)
        info_label.config(text=f"Book '{book.get_title()}' added successfully!")
        for e in [title_entry, author_entry, pages_entry, category_entry]:
            e.delete(0, 'end')

    create_button(root, "Add Book", add_book_action)
    create_button(root, "Back", welcome_screen)

def remove_book_screen():
    for w in root.winfo_children():
        w.destroy()
    create_label(root, "Remove Book", font_size=36, pady=30)
    create_label(root, "Enter book title to remove:", font_size=20)
    title_entry = create_entry(root)
    info_label = create_label(root, "", font_size=20)

    def remove_book_action():
        title = title_entry.get().strip()
        found_books = [b for b in system.get_books() if b.get_title() == title]

        if not found_books:
            info_label.config(text="Book not found!")
            return

        for b in found_books[:]:
            system.remove_book(b)
        info_label.config(text=f"Removed {len(found_books)} book(s)!")
        title_entry.delete(0, 'end')

    create_button(root, "Remove Book", remove_book_action)
    create_button(root, "Back", welcome_screen)

def categories_screen():
    for w in root.winfo_children():
        w.destroy()
    user = system.get_logged_in_user()
    create_label(root, "Categories", font_size=36, pady=20)
    create_button(root, "Search Book", search_screen, width=15)

    books_to_show = system.get_books()
    categories = {}
    for book in books_to_show:
        categories.setdefault(book.get_category(), []).append(book)

    side_frame = tk.Frame(root, bg=BG_COLOR)
    side_frame.pack(side="left", fill="y", padx=20)

    content_frame = tk.Frame(root, bg=BG_COLOR)
    content_frame.pack(side="left", fill="both", expand=True, padx=20)

    canvas = tk.Canvas(content_frame, bg=BG_COLOR, highlightthickness=0)
    scrollbar = tk.Scrollbar(content_frame, command=canvas.yview)
    scrollbar.pack(side="right", fill="y")
    canvas.pack(side="left", fill="both", expand=True)
    canvas.config(yscrollcommand=scrollbar.set)

    books_frame = tk.Frame(canvas, bg=BG_COLOR)
    canvas.create_window((0, 0), window=books_frame, anchor="nw")

    def show_books(cat):
        for w in books_frame.winfo_children():
            w.destroy()

        for b in categories[cat]:
            # Get user's progress for this book
            progress = None
            if isinstance(user, User):
                progress = user.get_progress_for_book(b.get_title())
            
            current_page = progress.get_current_page() if progress else 0
            display_text = f"{b.get_title()} ({current_page}/{b.get_total_pages()})"
            create_button(books_frame, display_text,
                         lambda book=b: book_info_screen(book), width=30, bg_color=LABEL_COLOR)

        books_frame.update_idletasks()
        canvas.config(scrollregion=canvas.bbox("all"))

    for cat in categories:
        count = len(categories[cat])
        create_button(side_frame, f"{cat} ({count})", lambda c=cat: show_books(c), width=15, bg_color=LABEL_COLOR)

    create_button(root, "Back", welcome_screen).pack(side="bottom")

def search_screen():
    for w in root.winfo_children():
        w.destroy()
    create_label(root, "Search Book", font_size=30, pady=20)
    entry = create_entry(root)
    result_frame = tk.Frame(root, bg=BG_COLOR)
    result_frame.pack(fill="both", expand=True)

    def do_search():
        for w in result_frame.winfo_children():
            w.destroy()
        query = entry.get().lower()
        found = False
        user = system.get_logged_in_user()

        for b in system.get_books():
            if (query in b.get_title().lower() or
                query in b.get_author().lower() or
                query in b.get_category().lower()):
                create_button(result_frame, f"{b.get_title()} by {b.get_author()}",
                            lambda book=b: book_info_screen(book), width=30,
                            bg_color=LABEL_COLOR)
                found = True

        if not found:
            create_label(result_frame, "No book found.", font_size=20)

    create_button(root, "Search", do_search)
    create_button(root, "Back", categories_screen)

def book_info_screen(book):
    for w in root.winfo_children():
        w.destroy()
    user = system.get_logged_in_user()
    
    create_label(root, f"{book.get_title()}", font_size=30, pady=10)
    create_label(root, f"Author: {book.get_author()}", font_size=22, pady=5)

    if not book.get_ratings_history():
        create_label(root, "Average Rating: No ratings yet", font_size=22, pady=5)
    else:
        avg = round(book.average_rating())
        stars = "★" * avg + "☆" * (5 - avg)
        create_label(root, f"Average Rating: {stars}", font_size=22, pady=5)

    create_label(root, f"Pages: {book.get_total_pages()}", font_size=20, pady=5)
    create_label(root, f"Category: {book.get_category()}", font_size=20, pady=5)

   
    if isinstance(user, User):
        progress = user.get_progress_for_book(book.get_title())
        if progress:
            create_label(root, f"Your progress: {progress.get_current_page()}/{book.get_total_pages()} pages", 
                        font_size=18, pady=5)

    def start_reading_handler():
        if isinstance(user, User):
            progress = user.start_reading_book(book)
            start_reading(book, user)
        else:
            start_reading(book, user)

    create_button(root, "Start Reading", start_reading_handler)
    create_button(root, "Back", categories_screen)

def start_reading(book, user):
    for w in root.winfo_children():
        w.destroy()
    
  
    progress = None
    if isinstance(user, User):
        progress = user.get_progress_for_book(book.get_title())
    
    current_page = progress.get_current_page() if progress else 0
    total_pages = book.get_total_pages()
    
    page_label = create_label(root, f"Page {current_page}/{total_pages}", font_size=30, pady=20)

    def next_page_action():
        if progress:
            finished = progress.next_page()
            page_label.config(text=f"Page {progress.get_current_page()}/{total_pages}")
            if finished:
                rating_screen(book)

    def prev_page_action():
        if progress:
            progress.previous_page()
            page_label.config(text=f"Page {progress.get_current_page()}/{total_pages}")

    create_button(root, "Next Page", next_page_action, width=20)
    create_button(root, "Previous Page", prev_page_action, width=20)
    create_button(root, "Back", lambda: book_info_screen(book), width=20)

def continue_reading(user, book):
    start_reading(book, user)

def rating_screen(book):
    for w in root.winfo_children():
        w.destroy()
    create_label(root, f"Rate '{book.get_title()}'", font_size=30, pady=30)
    stars_frame = tk.Frame(root, bg=BG_COLOR)
    stars_frame.pack(pady=20)
    rating_var = tk.IntVar()
    for i in range(1, 6):
        tk.Radiobutton(stars_frame, text=str(i), variable=rating_var, value=i,
                       font=("Helvetica", 24), bg=BG_COLOR, fg="white",
                       selectcolor=BG_COLOR).pack(side="left", padx=10)

    def submit_rating():
        r = rating_var.get()
        if r:
            book.rate_book(r)
            thanks_screen(book)

    create_button(root, "Submit", submit_rating, width=15)

def thanks_screen(book):
    for w in root.winfo_children():
        w.destroy()
    create_label(root, f"Thanks for rating '{book.get_title()}'!", font_size=30, pady=50)
    create_button(root, "Try Another Book", categories_screen, width=20)

login_screen()
root.mainloop()